# Case study: pricing a pension buy-in for an Italian pension fund

**Business question.** The board of an Italian occupational pension fund with about 1,500 pensioners in payment wants to insure its
longevity and investment risk through a **buy-in**: a single premium paid to a life insurer, which then pays the pensions. The insurer's
pricing actuary must answer:

1. What is the best estimate of the liabilities, with which mortality and discount basis?
2. How much capital does the business consume under the Solvency II standard formula, and what is the risk margin?
3. Which premium covers the technical provisions and pays the shareholders' target return on capital?
4. How does the standard formula compare with an internal-model view (one-year longevity trend risk and idiosyncratic risk)?
5. Which assumptions move the price most?

**Data and basis.** Mortality from **Eurostat** deaths and population for Italy (Poisson Lee-Carter by sex, pandemic years excluded),
scaled by a basis adjustment for pensioners; discount rates from the **ECB AAA curve** extrapolated with **Smith-Wilson** to an
ultimate forward rate of 3.30%. The membership is synthetic and illustrative (set `BUYIN_MEMBERSHIP_CSV` to a file with columns
`sex`, `age`, `pension` to use real data, which must never be committed).

**Conventions.** Pensions are paid yearly in advance while the member is alive; valuation at 31 December of the last mortality year;
amounts in euro. Solvency II references: Delegated Regulation (EU) 2015/35, Articles 39 (cost of capital), 138 (longevity),
140 (expense), 166-167 (interest rate), 204 (operational risk); Directive 2009/138/EC, Annex IV (correlations).

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "longevity_risk").is_dir())
try:
    import longevity_risk
except ImportError:  # not installed: use the source folder (the extension must be built in place)
    sys.path.insert(0, str(ROOT / "scripts" / "longevity_risk"))
    import longevity_risk
from longevity_risk import curves, data, lee_carter as lc, life_table as lt, portfolio, simulation as sim, solvency, synthetic

core = longevity_risk.require_cpp()

DATA_MODE = os.environ.get("LONGEVITY_RISK_DATA_MODE", "official")  # "official" (Eurostat + ECB) or "synthetic"
MEMBERSHIP_CSV = os.environ.get("BUYIN_MEMBERSHIP_CSV")               # optional real data: columns sex, age, pension
COUNTRY, AGES, FIRST_YEAR, PANDEMIC_YEARS = "IT", range(50, 100), 1975, (2020, 2021, 2022)
BASIS_ADJUSTMENT = 0.90        # pensioner mortality as a share of population mortality (socio-economic selection)
INDEXATION = 0.0               # annual pension increase
EXPENSE_PER_MEMBER = 40.0      # EUR per member per year, growing with EXPENSE_INFLATION
EXPENSE_INFLATION = 0.02
UFR, LLP = 0.033, 20
HURDLE_RATE = 0.10             # shareholders' target return on the capital held
N_SCENARIOS = 50_000
SEED = 20240101
OUTPUT_DIR = ROOT / "outputs" / "buy_in_pricing"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"longevity_risk {longevity_risk.__version__} | data mode: {DATA_MODE} | threads: {os.cpu_count()}")

## 1. Membership

In [ ]:
members = pd.read_csv(MEMBERSHIP_CSV) if MEMBERSHIP_CSV else synthetic.pensioners(n=1500)
print("Source:", members.attrs.get("source", MEMBERSHIP_CSV))
summary = members.groupby("sex").agg(members=("age", "size"), average_age=("age", "mean"),
                                     total_pension=("pension", "sum"), average_pension=("pension", "mean"))
display(summary)
top = members["pension"].sort_values(ascending=False)
print(f"Total annual pensions EUR {members['pension'].sum():,.0f}; the largest 5% of pensions are "
      f"{top.iloc[: len(top) // 20].sum() / top.sum():.0%} of the total (concentration)")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for sex, color in (("M", "C0"), ("F", "C3")):
    sub = members[members["sex"] == sex]
    axes[0].hist(sub["age"], bins=range(60, 97), alpha=0.6, color=color, label=sex)
    axes[1].hist(sub["pension"] / 1e3, bins=60, alpha=0.6, color=color, label=sex)
axes[0].set(title="Age distribution", xlabel="age")
axes[1].set(title="Annual pension", xlabel="EUR thousand")
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "membership.png", dpi=150)

## 2. Mortality basis

A Poisson Lee-Carter model is fitted by sex on ages 50-99; the pricing basis multiplies population death rates by the basis adjustment
($\ln m_{x,t} = a_x + \ln(0.90) + b_x k_t$). In practice the adjustment is estimated from the fund's own experience or from insured-lives
studies; it is one of the most material pricing assumptions and is tested in section 8.

In [ ]:
models, basis = {}, {}
for sex in ("M", "F"):
    if DATA_MODE == "official":
        D, E = data.load_deaths_exposures(COUNTRY, sex, AGES)
    else:
        D, E, _ = synthetic.deaths_and_exposures(AGES, range(1975, 2024), seed=7 if sex == "M" else 8,
                                                 drift=-1.2 if sex == "M" else -1.0, log_level=0.0 if sex == "M" else -0.4)
    keep = [y for y in D.columns if y >= FIRST_YEAR]
    fit = lc.fit_poisson(D.loc[:, keep], E.loc[:, keep], fit_years=[y for y in keep if y not in PANDEMIC_YEARS])
    pop = sim.ProjectionModel.from_fit(fit)
    models[sex] = sim.ProjectionModel(pop.age_min, pop.ax + np.log(BASIS_ADJUSTMENT), pop.bx, pop.rw)
    basis[sex] = {"calibration": f"{pop.rw.first_year}-{pop.rw.last_year}", "drift": pop.rw.drift, "sigma": pop.rw.sigma,
                  "cohort e65 (basis)": lt.curtate_life_expectancy(models[sex].central_q(65)) + 0.5,
                  "cohort e65 (population)": lt.curtate_life_expectancy(pop.central_q(65)) + 0.5}
print("Source:", D.attrs.get("source"))
T = models["M"].valuation_year
display(pd.DataFrame(basis).T)
print(f"Valuation date: 31 December {T}")

## 3. Discount curve

In [ ]:
if DATA_MODE == "official":
    svensson = data.load_ecb_svensson_parameters(f"{T}-12-01", f"{T}-12-31")
else:
    svensson = synthetic.svensson_parameters(f"{T}-12-31")
sw = curves.smith_wilson_from_svensson(curves.SvenssonCurve.from_series(svensson.iloc[-1]), UFR, LLP)
HORIZON = 120 - int(members["age"].min()) + 2
times = np.arange(HORIZON)
zero = np.r_[0.0, sw.zero_rate(times[1:])]           # annual compounding
discount = solvency.discount_factors(zero, times)
print("Curve:", svensson.attrs.get("source"), f"| Smith-Wilson alpha {sw.alpha:.3f}")
print("Zero rates (%):", {int(t): round(100 * float(zero[t]), 2) for t in (1, 5, 10, 20, 30, 50)})

## 4. Cash flows and best estimate

Expected pensions by year (central Lee-Carter projection on the pricing basis), expected expenses (EUR 40 per member per year growing at
2%), best estimate and Macaulay duration.

In [ ]:
cf = portfolio.expected_cash_flows(models, members, HORIZON, INDEXATION)
survivors = portfolio.expected_cash_flows(models, members.assign(pension=1.0), HORIZON)["total"]
exp_cf = EXPENSE_PER_MEMBER * survivors * (1 + EXPENSE_INFLATION) ** times
be_benefits = float(cf["total"] @ discount)
be_expenses = float(exp_cf @ discount)
best_estimate = be_benefits + be_expenses
duration = float((times * (cf["total"] + exp_cf)) @ discount / best_estimate)
print(f"Best estimate: benefits EUR {be_benefits:,.0f} + expenses EUR {be_expenses:,.0f} = EUR {best_estimate:,.0f}")
print(f"Macaulay duration {duration:.1f} years; best estimate / annual pensions = {be_benefits / members['pension'].sum():.2f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(times, cf["M"] / 1e6, label="men")
ax.bar(times, cf["F"] / 1e6, bottom=cf["M"] / 1e6, label="women")
ax.plot(times, exp_cf / 1e6, "k-", lw=1, label="expenses")
ax.set(xlim=(-0.5, 50), xlabel="year", ylabel="EUR million", title="Expected cash flows")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "cash_flows.png", dpi=150)

## 5. Solvency capital requirement (standard formula)

- **Longevity** (Art. 138): permanent 20% decrease of mortality rates.
- **Expense** (Art. 140): expenses +10% and expense inflation +1 percentage point.
- Life module: longevity and expense aggregated with correlation 0.25.
- **Interest rate** (Arts. 166-167): relative up and down shocks of the zero rates by maturity. The liability-only figure is shown for
  information; the insurer invests the premium in a duration- and cash-flow-matched bond portfolio, so the net market SCR is set to zero
  in the base case (the residual mismatch is a separate investment decision).
- Basic SCR with market-life correlation 0.25, plus operational risk (min(30% BSCR, 0.45% of technical provisions)).

In [ ]:
shocked_cf = portfolio.expected_cash_flows(models, members, HORIZON, INDEXATION, q_multiplier=1 - solvency.LONGEVITY_SHOCK)
shocked_survivors = portfolio.expected_cash_flows(models, members.assign(pension=1.0), HORIZON,
                                                  q_multiplier=1 - solvency.LONGEVITY_SHOCK)["total"]
scr_longevity_benefits = float(shocked_cf["total"] @ discount) - be_benefits
scr_longevity = scr_longevity_benefits + float(
    (EXPENSE_PER_MEMBER * shocked_survivors * (1 + EXPENSE_INFLATION) ** times) @ discount) - be_expenses
exp_shocked = 1.1 * EXPENSE_PER_MEMBER * survivors * (1 + EXPENSE_INFLATION + 0.01) ** times
scr_expense = float(exp_shocked @ discount) - be_expenses
scr_life = float(np.sqrt(scr_longevity**2 + scr_expense**2 + 2 * 0.25 * scr_longevity * scr_expense))
ir = solvency.interest_rate_scr(cf["total"] + exp_cf, times, zero)
scr_market = 0.0
bscr = solvency.basic_scr(scr_market, scr_life)
scr_table = pd.Series({"longevity": scr_longevity, "expense": scr_expense, "life (diversified)": scr_life,
                       "interest rate, liabilities only (information)": ir["scr"],
                       "market, net of matched assets": scr_market, "basic SCR": bscr})
display((scr_table / 1e6).rename("EUR million").to_frame().assign(**{"% of BE": 100 * scr_table / best_estimate}))

## 6. Risk margin and technical provisions

Cost-of-capital method (6%) on the SCR of non-hedgeable risks (longevity, expense, operational), projected in proportion to the best
estimate run-off; technical provisions = best estimate + risk margin. The operational SCR depends on the technical provisions, so the two
are solved by fixed-point iteration.

In [ ]:
runoff = solvency.best_estimate_runoff(cf["total"] + exp_cf, discount)
tp = best_estimate
for _ in range(20):
    scr_op = solvency.operational_scr(bscr, tp)
    rm = solvency.risk_margin(scr_life + scr_op, runoff, discount)
    tp = best_estimate + rm
scr_total = bscr + scr_op
print(f"Operational SCR EUR {scr_op:,.0f}; total SCR EUR {scr_total:,.0f} ({100 * scr_total / best_estimate:.2f}% of BE)")
print(f"Risk margin EUR {rm:,.0f} ({100 * rm / best_estimate:.2f}% of BE); technical provisions EUR {tp:,.0f}")

## 7. Premium

The risk margin pays 6% on the capital for non-hedgeable risks. Shareholders require a hurdle rate of 10% on all the capital held over
the run-off, so the profit loading adds $(h - 6\%)\sum_t \text{SCR}(t)\,v(t+1)$ with $\text{SCR}(t)$ projected in proportion to the best
estimate. The fund's book value uses a period table (mortality frozen at year $T$, no future improvements) on the same curve, a common
simplification in pension-fund accounting that explains part of the gap between book value and buy-in price.

In [ ]:
capital_annuity = solvency.risk_margin(scr_total, runoff, discount, coc=1.0)   # sum_t SCR(t) v(t+1)
profit_loading = (HURDLE_RATE - solvency.COST_OF_CAPITAL) * capital_annuity
premium = tp + profit_loading

period_models = {s: sim.ProjectionModel(m.age_min, m.ax, m.bx,
                                        lc.RandomWalkDrift(0.0, 0.0, 0.0, m.rw.k_last, m.rw.k_last, m.rw.first_year, m.rw.last_year))
                 for s, m in models.items()}
book_value = float(portfolio.expected_cash_flows(period_models, members, HORIZON, INDEXATION)["total"] @ discount)

waterfall = pd.Series({"best estimate, benefits": be_benefits, "best estimate, expenses": be_expenses, "risk margin": rm,
                       "profit loading": profit_loading})
quote = pd.DataFrame({"EUR": waterfall, "% of BE": 100 * waterfall / best_estimate})
quote.loc["premium"] = [premium, 100 * premium / best_estimate]
display(quote)
print(f"Fund's book value on a period table: EUR {book_value:,.0f}; premium / book value = {premium / book_value:.3f}")

fig, ax = plt.subplots(figsize=(9, 3.8))
left = 0.0
for label, value in waterfall.items():
    ax.barh(0, value / 1e6, left=left / 1e6, label=label)
    left += value
ax.axvline(book_value / 1e6, color="k", ls="--", label="fund's book value (period table)")
ax.set(yticks=[], xlabel="EUR million", title=f"Buy-in premium EUR {premium / 1e6:,.1f} million")
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "premium_waterfall.png", dpi=150)

## 8. Internal-model cross-check

- **One-year longevity trend risk** (Richards, Currie and Ritchie, 2014), computed in C++ for the whole membership with the same
  scenarios for every member and both sexes: 99.5% quantile of the value after one year of new data minus the best estimate.
- **Run-off risk including idiosyncratic risk**: 50,000 scenarios, each with simulated lifetimes of all members, showing how much the
  concentration in large pensions adds to trend risk.

In [ ]:
t0 = time.perf_counter()
be_model, one_year = portfolio.one_year_values(models, members, discount, N_SCENARIOS, SEED, INDEXATION)
runoff_sim = portfolio.simulate(models, members, discount, N_SCENARIOS, idiosyncratic=True, seed=SEED, indexation=INDEXATION)
elapsed = time.perf_counter() - t0
var_1y = np.quantile(one_year, 0.995) - be_model
q_sys = np.quantile(runoff_sim["systematic"], 0.995) - runoff_sim["systematic"].mean()
q_all = np.quantile(runoff_sim["realised"], 0.995) - runoff_sim["realised"].mean()
comparison = pd.Series({"standard formula longevity SCR (benefits)": scr_longevity_benefits,
                        "one-year VaR 99.5% (trend)": var_1y,
                        "run-off 99.5%: systematic only": q_sys,
                        "run-off 99.5%: systematic + idiosyncratic": q_all})
display(pd.DataFrame({"EUR": comparison, "% of benefit BE": 100 * comparison / be_benefits}))
print(f"{2 * N_SCENARIOS:,} portfolio scenarios ({len(members):,} members, simulated lifetimes included) in {elapsed:.1f} s (C++)")

## 9. Sensitivities

In [ ]:
def price(basis_adj=BASIS_ADJUSTMENT, indexation=INDEXATION, ufr=UFR, drift_scale=1.0, coc=solvency.COST_OF_CAPITAL,
          hurdle=HURDLE_RATE):
    ms = {}
    for s, m in models.items():
        rw = lc.RandomWalkDrift(m.rw.drift * drift_scale, m.rw.sigma, m.rw.drift_se, m.rw.k_first, m.rw.k_last,
                                m.rw.first_year, m.rw.last_year)
        ms[s] = sim.ProjectionModel(m.age_min, m.ax - np.log(BASIS_ADJUSTMENT) + np.log(basis_adj), m.bx, rw)
    c = sw if ufr == UFR else curves.smith_wilson_from_svensson(curves.SvenssonCurve.from_series(svensson.iloc[-1]), ufr, LLP)
    z = np.r_[0.0, c.zero_rate(times[1:])]
    v = solvency.discount_factors(z, times)
    alive = portfolio.expected_cash_flows(ms, members.assign(pension=1.0), HORIZON)["total"]
    expenses = EXPENSE_PER_MEMBER * alive * (1 + EXPENSE_INFLATION) ** times
    flows = portfolio.expected_cash_flows(ms, members, HORIZON, indexation)["total"] + expenses
    be = float(flows @ v)
    alive_shocked = portfolio.expected_cash_flows(ms, members.assign(pension=1.0), HORIZON, q_multiplier=0.8)["total"]
    shocked = (portfolio.expected_cash_flows(ms, members, HORIZON, indexation, 0.8)["total"]
               + EXPENSE_PER_MEMBER * alive_shocked * (1 + EXPENSE_INFLATION) ** times)
    longevity = float(shocked @ v) - be
    life = np.sqrt(longevity**2 + scr_expense**2 + 2 * 0.25 * longevity * scr_expense)
    run = solvency.best_estimate_runoff(flows, v)
    op = solvency.operational_scr(life, be)
    rm_ = solvency.risk_margin(life + op, run, v, coc)
    return be + rm_ + (hurdle - coc) * solvency.risk_margin(life + op, run, v, 1.0)

base = price()
cases = {"basis adjustment 0.85": dict(basis_adj=0.85), "basis adjustment 1.00": dict(basis_adj=1.00),
         "improvements -25%": dict(drift_scale=0.75), "improvements +25%": dict(drift_scale=1.25),
         "indexation 1%": dict(indexation=0.01), "indexation 2%": dict(indexation=0.02),
         "UFR 3.00%": dict(ufr=0.030), "hurdle rate 12%": dict(hurdle=0.12)}
sens = pd.Series({k: 100 * (price(**v) / base - 1) for k, v in cases.items()}, name="premium change (%)")
display(sens.to_frame())
fig, ax = plt.subplots(figsize=(8, 3.8))
sens.sort_values().plot.barh(ax=ax, color=["C3" if x < 0 else "C0" for x in sens.sort_values()])
ax.set(xlabel="% change of the premium", title="Premium sensitivities")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "sensitivities.png", dpi=150)

## 10. Summary for the pension fund board

In [ ]:
assumptions = sens.drop(index=[k for k in sens.index if k.startswith("indexation")])   # product features excluded
lines = [
    f"Membership: {len(members):,} pensioners, annual pensions EUR {members['pension'].sum() / 1e6:,.1f} million.",
    f"Best estimate EUR {best_estimate / 1e6:,.1f} million (duration {duration:.1f} years) on the cohort Lee-Carter basis "
    f"with {BASIS_ADJUSTMENT:.0%} of population mortality and the Smith-Wilson curve (UFR {UFR:.2%}).",
    f"Standard-formula SCR EUR {scr_total / 1e6:,.1f} million ({100 * scr_total / best_estimate:.1f}% of BE), driven by longevity; "
    f"risk margin {100 * rm / best_estimate:.1f}% of BE.",
    f"Indicative premium EUR {premium / 1e6:,.1f} million = {100 * premium / best_estimate:.1f}% of BE and "
    f"{premium / book_value:.2f} times the fund's period-table book value (the gap is mostly future mortality improvements).",
    f"Internal view: one-year 99.5% trend risk is {100 * var_1y / be_benefits:.1f}% of the benefit BE, against "
    f"{100 * comparison.iloc[0] / be_benefits:.1f}% for the standard-formula shock.",
    f"Most material pricing assumption: {assumptions.abs().idxmax()} ({assumptions[assumptions.abs().idxmax()]:+.1f}% on the premium); "
    f"indexing pensions by 2% a year would raise the premium by {sens['indexation 2%']:+.1f}%.",
]
print("\n".join(lines))
_ = (OUTPUT_DIR / "board_summary.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

### Limitations

- Illustrative membership; real quotes use the fund's data, its own mortality experience and bespoke expense assumptions.
- Population mortality with a flat adjustment; no cohort or socio-economic segmentation by pension amount (richer members live longer,
  which would increase the price for this concentrated portfolio).
- Matched assets assumed: spread, reinvestment and credit risks of the backing portfolio, and the matching adjustment, are not modelled.
- Risk-margin projection by the best-estimate proxy; the Solvency II review (Directive (EU) 2025/2) lowers the cost-of-capital rate and
  introduces a time-dependent factor, which would reduce the risk margin once applicable.

### References

- Commission Delegated Regulation (EU) 2015/35, Articles 37-39, 138, 140, 166, 167, 204; Directive 2009/138/EC, Annex IV.
- EIOPA, Guidelines on valuation of technical provisions (EIOPA-BoS-14/166), guideline on simplifications for the risk margin.
- Richards, S. J., Currie, I. D. and Ritchie, G. P. (2014). A value-at-risk framework for longevity trend risk. *British Actuarial Journal*, 19(1), 116-139.
- Brouhns, N., Denuit, M. and Vermunt, J. K. (2002). A Poisson log-bilinear regression approach to the construction of projected lifetables. *Insurance: Mathematics and Economics*, 31(3), 373-393.
- Eurostat (`demo_magec`, `demo_pjan`); European Central Bank, euro area yield curves.